In [52]:
posts_path =r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts.csv"
comments_path = r"C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\data\processed\posts_comments.csv"

In [53]:
import pandas as pd
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
from collections import Counter


In [54]:
post_df = pd.read_csv(posts_path)
comment_df = pd.read_csv(comments_path, sep = ';')

In [55]:
comment_agg = comment_df.groupby('post_id')['comments'].apply(list).reset_index()
merged_df = post_df.merge(comment_agg, on='post_id', how='left')

In [56]:
df = merged_df.drop(columns=['image'])
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount,comments
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142,"[I have a proposal for u, send me a DM please!..."
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",NaN,6565,86,[Hey Tina I just want to make sure you got my ...
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,NaN,28936,123,"[love this monochrome look :fire::fire::fire:,..."
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,CHANELFallWinter,4764,215,[:OK_hand_medium_skin_tone::white_heart::smili...
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,CannesFilmFestival,13379,123,"[Маша, ты королева :broken_heart:, :heart_on_f..."


In [57]:
print ("Rows : " ,df.shape[0])
print ("Columns : " ,df.shape[1])
print ("\nFeatures : \n" ,df.columns.tolist())

Rows :  827
Columns :  8

Features : 
 ['post_id', 'timestamp', 'ownerUsername', 'caption', 'hashtags', 'likesCount', 'commentsCount', 'comments']


In [58]:
missing_values = df.isnull().sum()
print(missing_values)

post_id            0
timestamp          0
ownerUsername      0
caption            7
hashtags         353
likesCount         0
commentsCount      0
comments           0
dtype: int64


In [59]:
# Fill missing value in caption and hashtags column by ""
df['caption'] = df['caption'].fillna('')
df['hashtags'] = df['hashtags'].fillna('')

In [60]:
df.isnull().sum()

post_id          0
timestamp        0
ownerUsername    0
caption          0
hashtags         0
likesCount       0
commentsCount    0
comments         0
dtype: int64

In [61]:
df.dtypes

post_id           int64
timestamp        object
ownerUsername    object
caption          object
hashtags         object
likesCount        int64
commentsCount     int64
comments         object
dtype: object

### "Caption" Column

In [62]:
# Clean caption
import re

def clean_caption(text):
    if pd.isna(text) or text == 'NaN':
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\s\U0001F000-\U0001F9FF]', ' ', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_caption'] = df['caption'].apply(clean_caption)

In [63]:
# Process Emoji in caption
import emoji

def process_emoji(text):
    emojis_list = [c for c in text if c in emoji.EMOJI_DATA]
    
    emoji_descriptions = []
    for e in emojis_list:
        emoji_name = emoji.demojize(e).replace(':', '').replace('_', '   ')
        emoji_descriptions.append(emoji_name)
    
    text_without_emoji = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    return text_without_emoji, emoji_descriptions

df[['caption_no_emoji', 'emoji_descriptions']] = df['clean_caption'].apply(lambda x: pd.Series(process_emoji(x)))


### "Hashtag" column

In [64]:
# Process hashtags from hashtag column
def process_hashtags_from_column(hashtag_text):
    if pd.isna(hashtag_text) or hashtag_text == 'nan' or hashtag_text == '':
        return []
    
    if hashtag_text.startswith('[') and hashtag_text.endswith(']'):
        hashtag_text = hashtag_text[1:-1]
        
        tags = [tag.strip().strip("'").strip('"') for tag in hashtag_text.split(',')]
    else:
        tags = [tag.strip().strip('#') for tag in re.split(r'[,#]', hashtag_text) if tag.strip()]
    
    processed_hashtags = []
    for tag in tags:
        if tag:
            words = re.findall(r'[A-Z]?[a-z]+|[A-Z]+(?=[A-Z]|$)', tag)
            if not words:
                words = [tag]
            processed_hashtags.extend([word.lower() for word in words])
    
    return processed_hashtags

df['hashtags'] = df['hashtags'].apply(process_hashtags_from_column)

In [65]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')


# Tokenize and remove stopwords
def tokenize_and_remove_stopwords(text):
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words and word not in string.punctuation]
    
    return filtered_tokens

df['caption_tokens'] = df['caption_no_emoji'].apply(tokenize_and_remove_stopwords)
df['combined_tokens'] = df.apply(lambda row: row['caption_tokens'] + row['emoji_descriptions'] + row['hashtags'], axis=1)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [66]:
df.head()

,post_id,timestamp,ownerUsername,caption,hashtags,likesCount,commentsCount,comments,clean_caption,caption_no_emoji,emoji_descriptions,caption_tokens,combined_tokens
0,1,2023-06-02 16:34:43+00:00,maryleest,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,"[kilian, cannes]",32070,142,"[I have a proposal for u, send me a DM please!...",cannes 2023 with kilianparis 🤍 kiliancannes we...,cannes 2023 with kilianparis kiliancannes wea...,[white heart],"[cannes, 2023, kilianparis, kiliancannes, wear...","[cannes, 2023, kilianparis, kiliancannes, wear..."
1,2,2023-01-01 07:49:16+00:00,tinaabeysekara,"As the clock struck midnight to ring in 2022, ...",[],6565,86,[Hey Tina I just want to make sure you got my ...,as the clock struck midnight to ring in 2022 i...,as the clock struck midnight to ring in 2022 i...,"[woman dancing, medium skin tone, dizzy,...","[clock, struck, midnight, ring, 2022, cried, w...","[clock, struck, midnight, ring, 2022, cried, w..."
2,3,2023-05-29 18:57:51+00:00,maryleest,The famous stairs 🤎 Photo @gustave_durin Dre...,[],28936,123,"[love this monochrome look :fire::fire::fire:,...",the famous stairs 🤎 photo gustave_durin dress ...,the famous stairs photo gustave_durin dress m...,[brown heart],"[famous, stairs, photo, gustave_durin, dress, ...","[famous, stairs, photo, gustave_durin, dress, ..."
3,4,2023-03-07 19:30:42+00:00,stephaniebroek,I visualized this moment so many times before....,"[chanel, fall, winter]",4764,215,[:OK_hand_medium_skin_tone::white_heart::smili...,i visualized this moment so many times before ...,i visualized this moment so many times before ...,"[white heart, black heart]","[visualized, moment, many, times, first, chane...","[visualized, moment, many, times, first, chane..."
4,5,2023-05-23 21:11:12+00:00,maryleest,Got to witness such historical moment in cinem...,"[cannes, film, festival]",13379,123,"[Маша, ты королева :broken_heart:, :heart_on_f...",got to witness such historical moment in cinem...,got to witness such historical moment in cinem...,[broken heart],"[got, witness, historical, moment, cinema, kil...","[got, witness, historical, moment, cinema, kil..."


### BERT Embedding for combined_tokens

In [67]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

c:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\venv\lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [68]:
def get_bert_embedding(tokens):
    """Convert combined_tokens (list of words) to a BERT embedding."""
    text = " ".join(tokens)  # Convert list to a single sentence
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()  # Take mean embedding

# Apply to dataframe
df['bert_embedding'] = df['combined_tokens'].apply(get_bert_embedding)

print(df[['combined_tokens', 'bert_embedding']].head())

                                     combined_tokens  \
0  [cannes, 2023, kilianparis, kiliancannes, wear...   
1  [clock, struck, midnight, ring, 2022, cried, w...   
2  [famous, stairs, photo, gustave_durin, dress, ...   
3  [visualized, moment, many, times, first, chane...   
4  [got, witness, historical, moment, cinema, kil...   

                                      bert_embedding  
0  [-0.05747311, -0.37669456, 0.27712598, 0.08630...  
1  [0.07370757, 0.0074080415, 0.94746697, -0.1916...  
2  [0.04359917, 0.13778722, 0.13084672, -0.280214...  
3  [-0.008063217, -0.07746659, 0.69791627, -0.024...  
4  [0.2409971, 0.20214732, 0.16738108, 0.00604014...  
